In [108]:
from pathlib import Path
notebook_directory = Path.cwd()

import numpy as np
import pandas as pd
import xlwings as xw

import statsmodels.api as sm

In [109]:
prices_csv = 'sectors'

start_date = pd.Timestamp("2026-03-25")
end_date   = pd.Timestamp("2026-06-14")

div_month = "202606"

output_price_filename = prices_csv + " " + div_month

In [110]:
group = 'Sector'
ishare_fee_diff = .003

In [111]:
prices_path = (
    notebook_directory.parent.parent
    / "backtesting"
    / "historical prices"
    / f"{prices_csv}.csv"
)
prices_df = pd.read_csv(prices_path, parse_dates=["date"])


db_path = (
    notebook_directory.parent
    / "spreadsheets"
    / "2026 Fin Inst Database.xlsx"
)
wb = xw.Book(db_path)
ws = wb.sheets["Scalar Inputs Table"]
db_df = ws.tables["scalar_inputs_table"].range.options(pd.DataFrame, header=1, index=False).value


anchor_path = (
    notebook_directory.parent
    / "spreadsheets"
    / "2026 Group Trading Inputs.xlsm"
)
wb = xw.Book(anchor_path)
ws = wb.sheets["ANCHOR INPUTs"]
anchor_df = ws.tables["anchor_inputs"].range.options(pd.DataFrame, header=1, index=False).value
anchor_df = anchor_df[anchor_df['group'] == group].copy()


In [112]:
#previous_month = (pd.Timestamp.today() - pd.DateOffset(months=1)).to_period("M")
#prices_df = prices_df.loc[prices_df["date"].dt.to_period("M").eq(previous_month)].copy()

prices_df["date"] = pd.to_datetime(prices_df["date"], errors="coerce")
prices_df = prices_df.loc[prices_df["date"].between(start_date, end_date)].copy()

In [113]:
anchor_df = anchor_df[anchor_df['group'] == group].copy()

In [114]:
symbols = anchor_df['symbol'].to_list()

In [115]:
start_date = prices_df['date'].iloc[0]
prices_df["days"] = (prices_df["date"] - start_date).dt.days

In [116]:
ishare_daily_log_rate = np.log1p(ishare_fee_diff) / 365
prices_df["ishare_scalar"] = np.exp(ishare_daily_log_rate * prices_df["days"])

In [117]:
ishare_symbols = [symbol for symbol in symbols if symbol.lower().startswith("i")]
for sym in ishare_symbols:
    prices_df[f'{sym}*'] = prices_df[sym] * prices_df["ishare_scalar"]

In [118]:
# div_month = (pd.Timestamp.today() - pd.DateOffset(months=1)).strftime("%Y%m")
div_column = f"div {div_month}"

for sym in symbols:
    dividends = db_df.loc[db_df["symbol"].eq(sym), div_column].to_list()
    div_amt = float(dividends[0])
    prices_df[f"{sym} div"] = div_amt

In [119]:
for sym in symbols:
    if sym in ishare_symbols:
        calc_sym = f'{sym}*'
    else:
        calc_sym = sym

    prices_df[f'{calc_sym} no div'] = prices_df[calc_sym] - prices_df[f'{sym} div']

In [120]:
for sym in symbols:
    calc_sym = f"{sym}*" if sym in ishare_symbols else sym

    anchor_matches = anchor_df.loc[
        anchor_df["symbol"].eq(sym), "anchor"
    ]

    if anchor_matches.empty:
        print(f"Skipping {sym}: no anchor found")
        continue

    anchor = anchor_matches.iloc[0]
    anchor = f"{anchor}*" if anchor in ishare_symbols else anchor

    y = prices_df[anchor]
    X = sm.add_constant(prices_df[calc_sym])

    model = sm.OLS(y, X).fit()

    anchor_df.loc[anchor_df['symbol'].eq(sym), 'intercept'] = model.params["const"]
    anchor_df.loc[anchor_df['symbol'].eq(sym), 'slope'] = model.params[calc_sym]

    # Regression using non-dividend-adjusted prices
    no_div_sym = f"{calc_sym} no div"
    no_div_anchor = f"{anchor} no div"

    y = prices_df[no_div_anchor]
    X = sm.add_constant(prices_df[no_div_sym])

    model = sm.OLS(y, X, missing="drop").fit()

    anchor_df.loc[anchor_df['symbol'].eq(sym), "intercept no div"] = model.params["const"]
    anchor_df.loc[anchor_df['symbol'].eq(sym), "slope no div"] = model.params[no_div_sym]

In [121]:
anchor_df = anchor_df.drop(columns="moving_avg_days")

In [122]:
wb = xw.Book(anchor_path)
ws = wb.sheets["SCALAR OUTPUTS"]
ws["A1"].options(index=False, header=True).value = anchor_df

In [123]:
prices_path = (
    notebook_directory.parent.parent
    / "backtesting"
    / "ratios"
    / f"{output_price_filename}.csv"
)

prices_df.to_csv(prices_path, index=False)